In [12]:
import pandas as pd

df_inp = pd.read_csv(r"D:\Coding\Data\Lanzhou_cfdc\processed\N_INP(202409-202509)v2.4.1.csv")
df_inp = df_inp[df_inp['Is_Significant'] == True]

target_temp = -30
df_temp = df_inp[df_inp['T_a(°C)'] == target_temp]
conc_mean = df_temp['N_inp_net(#/L)'].groupby(df_temp['season']).mean()
conc_std = df_temp['N_inp_net(#/L)'].groupby(df_temp['season']).std()
conc_count = df_temp['N_inp_net(#/L)'].groupby(df_temp['season']).count()
print(conc_mean)
print(conc_std)
print(conc_count)

season
Autumn    52.465797
Spring    88.296621
Summer    24.142866
Winter     7.586789
Name: N_inp_net(#/L), dtype: float64
season
Autumn     70.668668
Spring    103.119735
Summer     19.824069
Winter      5.332167
Name: N_inp_net(#/L), dtype: float64
season
Autumn    294
Spring    511
Summer    204
Winter    205
Name: N_inp_net(#/L), dtype: int64


In [10]:
df_temp

,Date,N_inp_net(#/L),T_inp(°C),SS_w,SS_i,Total_Sampling_Volume(L),Significance_Level(#/L),Is_Significant,season,T_a(°C)
0,2024-09-24 14:41:43.068586000,14.050137,-29.797332,5.761454,41.444960,6.149300,2.617877,True,Autumn,-30
1,2024-09-26 14:13:58.060174000,31.918336,-29.789597,5.983611,41.736518,6.291500,4.248877,True,Autumn,-30
2,2024-09-26 14:26:14.060563000,27.829487,-29.781749,5.968580,41.693702,6.266200,4.178253,True,Autumn,-30
3,2024-09-26 16:07:50.060424000,15.449528,-29.756955,4.246887,39.362124,6.032083,4.010179,True,Autumn,-30
4,2024-09-26 16:20:06.060029000,6.189200,-29.747240,4.200907,39.280730,6.042150,3.872997,True,Autumn,-30
...,...,...,...,...,...,...,...,...,...,...
2702,2025-09-29 21:33:33.133998000,26.282977,-30.085045,5.468698,41.433453,6.354783,3.552399,True,Autumn,-30
2703,2025-09-29 21:45:49.134215000,25.420881,-30.072571,5.592314,41.589166,6.370000,3.609625,True,Autumn,-30
2704,2025-09-29 21:58:05.133970000,26.181910,-30.079753,5.542714,41.523903,6.392333,3.694665,True,Autumn,-30
2705,2025-09-29 22:10:21.133372999,19.554389,-30.081190,5.423461,41.380286,6.389517,3.211531,True,Autumn,-30


In [19]:
import pandas as pd
import pingouin as pg
import numpy as np

df = pd.read_csv(r"D:\Coding\Data\Lanzhou_cfdc\processed\N_INP(202409-202509)v2.4.1.csv")
df = df[df['Is_Significant'] == True]
target_temp = -30
df_temp = df[df['T_a(°C)'] == target_temp]

# --- 步骤 1：检验方差齐性 (Levene's Test) ---
# 这一步是为了验证你的观察，确认方差确实不齐
levene_test = pg.homoscedasticity(data=df_temp, dv='N_inp_net(#/L)', group='season')
print("=== 方差齐性检验 (Levene's Test) ===")
print(levene_test)
print("注：如果 pval < 0.05，说明方差不齐，必须使用 Welch's ANOVA。\n")

# --- 步骤 2：Welch's ANOVA (适用于方差不齐的情况) ---
welch_anova = pg.welch_anova(dv='N_inp_net(#/L)', between='season', data=df_temp)
print("=== Welch's ANOVA 结果 ===")
print(welch_anova)
print("注：如果 p-unc < 0.05，说明四个季节的总体平均值至少有两个存在显著差异。\n")

# --- 步骤 3：事后检验 (Games-Howell Test) ---
# 如果 Welch's ANOVA 显著，我们需要知道具体是哪两个季节不同
# Games-Howell 是专门针对方差不齐和样本量可能不等的两两比较方法
if welch_anova['p_unc'].values[0] < 0.05:
    posthoc = pg.pairwise_gameshowell(dv='N_inp_net(#/L)', between='season', data=df_temp)
    print("=== 两两比较事后检验 (Games-Howell) ===")
    print(posthoc[['A', 'B', 'mean_A', 'mean_B', 'diff', 'pval']])
    print("注：pval < 0.05 表示这两个季节之间差异显著。")

=== 方差齐性检验 (Levene's Test) ===
                W          pval  equal_var
levene  62.618784  1.231355e-37      False
注：如果 pval < 0.05，说明方差不齐，必须使用 Welch's ANOVA。

=== Welch's ANOVA 结果 ===
   Source  ddof1       ddof2           F         p_unc       np2
0  season      3  519.477204  181.113836  2.315346e-80  0.149283
注：如果 p-unc < 0.05，说明四个季节的总体平均值至少有两个存在显著差异。

=== 两两比较事后检验 (Games-Howell) ===
        A       B     mean_A     mean_B       diff          pval
0  Autumn  Spring  52.465797  88.296621 -35.830824  4.913464e-08
1  Autumn  Summer  52.465797  24.142866  28.322931  1.507360e-09
2  Autumn  Winter  52.465797   7.586789  44.879008  0.000000e+00
3  Spring  Summer  88.296621  24.142866  64.153755  0.000000e+00
4  Spring  Winter  88.296621   7.586789  80.709833  5.950795e-14
5  Summer  Winter  24.142866   7.586789  16.556078  0.000000e+00
注：pval < 0.05 表示这两个季节之间差异显著。
